<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [1]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [2]:
# 1. Consolidación y limpieza de datos

# a) Consolidar archivos de años (concat) y normalizar a minúscula
dfs_anio = []
for archivo in archivos_anio:
    df_temp = pd.read_csv(archivo)
    df_temp.columns = df_temp.columns.str.lower()
    dfs_anio.append(df_temp)

df_anio = pd.concat(dfs_anio, ignore_index=True)
print("=" * 60)
print("a) df_anio consolidado")
print("=" * 60)
print(f"Filas: {df_anio.shape[0]} | Columnas: {df_anio.shape[1]}")
print(df_anio.head())

# b) Detectar código ISO asociado a dos nombres distintos
print("\n" + "=" * 60)
print("b) Inconsistencias en df_codigos")
print("=" * 60)
duplicados_codigo = df_codigos.groupby('codigo_iso').filter(lambda x: x['pais'].nunique() > 1)
print(duplicados_codigo)

# Eliminar registro inconsistente (conservamos el primero como válido)
df_codigos_limpio = df_codigos.drop_duplicates(subset='codigo_iso', keep='first').reset_index(drop=True)
print(f"\nFilas originales: {len(df_codigos)} | Filas tras limpieza: {len(df_codigos_limpio)}")

# c) Merge entre df_anio y df_codigos por codigo_iso (inner)
# El archivo df_anio tiene una columna 'pais' que conflictúa: usamos solo el código del país
# Primero necesitamos mapear el nombre del país a su código ISO

# Si df_anio tiene 'pais', creamos el código a partir del df_codigos_limpio
if 'codigo_iso' not in df_anio.columns:
    # Mapeamos a través del nombre de país
    mapa_pais_codigo = df_codigos_limpio.set_index('pais')['codigo_iso'].to_dict()
    df_anio['codigo_iso'] = df_anio['pais'].map(mapa_pais_codigo)
    df_anio = df_anio.drop(columns=['pais'])

df = pd.merge(df_anio, df_codigos_limpio, on='codigo_iso', how='inner')
print("\n" + "=" * 60)
print("c) df final (merge)")
print("=" * 60)
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
print(df.head())

a) df_anio consolidado
Filas: 3060 | Columnas: 4
  codigo_iso  anio  indice  ranking
0        AFG  2001    35.5     59.0
1        AGO  2001    30.2     50.0
2        ALB  2001     NaN      NaN
3        AND  2001     NaN      NaN
4        ARE  2001     NaN      NaN

b) Inconsistencias en df_codigos
    codigo_iso      pais
179        ZWE  Zimbabue
180        ZWE      malo

Filas originales: 181 | Filas tras limpieza: 180

c) df final (merge)
Filas: 3060 | Columnas: 5
  codigo_iso  anio  indice  ranking                    pais
0        AFG  2001    35.5     59.0             Afghanistán
1        AGO  2001    30.2     50.0                  Angola
2        ALB  2001     NaN      NaN                 Albania
3        AND  2001     NaN      NaN                 Andorra
4        ARE  2001     NaN      NaN  Emiratos Árabes Unidos




### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [3]:
# 2. Exploración inicial del conjunto de datos

print("=" * 60)
print("ESTRUCTURA DEL DATAFRAME")
print("=" * 60)
print(f"Filas (observaciones): {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"Nombres de columnas: {list(df.columns)}")
print("\nTipos de datos:")
print(df.dtypes)

print("\n" + "=" * 60)
print("INFO COMPLETA DEL DATAFRAME")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("RESUMEN ESTADÍSTICO")
print("=" * 60)
print(df.describe())

print("\n" + "=" * 60)
print("VALORES EXTREMOS DE INDICE")
print("=" * 60)
print(f"Mínimo: {df['indice'].min():.4f}")
print(f"Máximo: {df['indice'].max():.4f}")
print(f"Promedio: {df['indice'].mean():.4f}")

# País con menor índice (mayor libertad)
idx_min = df['indice'].idxmin()
print(f"\nPaís con menor indice: {df.loc[idx_min, 'pais']} ({df.loc[idx_min, 'anio']})")

# País con mayor índice (menor libertad)
idx_max = df['indice'].idxmax()
print(f"País con mayor indice: {df.loc[idx_max, 'pais']} ({df.loc[idx_max, 'anio']})")

print("\n" + "=" * 60)
print("DATOS FALTANTES")
print("=" * 60)
print("Cantidad de nulos por columna:")
print(df.isnull().sum())
print("\nProporción de nulos (%):")
print((df.isnull().sum() / len(df) * 100).round(2))

print("\n" + "=" * 60)
print("UNICIDAD Y DUPLICADOS")
print("=" * 60)
print(f"Países distintos: {df['pais'].nunique()}")
print(f"Años distintos: {df['anio'].nunique()}")
print(f"Rango de años: {df['anio'].min()} - {df['anio'].max()}")
print(f"Filas duplicadas: {df.duplicated().sum()}")

print("\n" + "=" * 60)
print("VALIDACIÓN CRUZADA: códigos ISO con múltiples nombres de país")
print("=" * 60)
inconsistencias = df.groupby('codigo_iso')['pais'].nunique()
inconsistencias = inconsistencias[inconsistencias > 1]
if len(inconsistencias) == 0:
    print("No hay inconsistencias: cada código ISO tiene un único país.")
else:
    print(inconsistencias)

ESTRUCTURA DEL DATAFRAME
Filas (observaciones): 3060
Columnas: 5
Nombres de columnas: ['codigo_iso', 'anio', 'indice', 'ranking', 'pais']

Tipos de datos:
codigo_iso     object
anio            int64
indice        float64
ranking       float64
pais           object
dtype: object

INFO COMPLETA DEL DATAFRAME
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3060 entries, 0 to 3059
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   codigo_iso  3060 non-null   object 
 1   anio        3060 non-null   int64  
 2   indice      2664 non-null   float64
 3   ranking     2837 non-null   float64
 4   pais        3060 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 119.7+ KB

RESUMEN ESTADÍSTICO
              anio        indice        ranking
count  3060.000000   2664.000000    2837.000000
mean   2009.941176    205.782316     477.930913
std       5.786024   2695.525264    6474.935347
min    2001.000000    




### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [4]:
# 3. Comparación regional: países latinoamericanos

america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']

# Filtrar países latinoamericanos y registros con índice válido
df_america = df.loc[df['codigo_iso'].isin(america) & df['indice'].notna()].copy()
print(f"Total de registros válidos para América: {len(df_america)}")
print(f"Países encontrados: {df_america['codigo_iso'].nunique()}")

# a) Enfoque con ciclo for
print("\n" + "=" * 60)
print("a) Enfoque con ciclo FOR")
print("=" * 60)
resultados_for = []
for anio in sorted(df_america['anio'].unique()):
    df_anio = df_america.loc[df_america['anio'] == anio]
    if len(df_anio) == 0:
        continue
    pais_min = df_anio.loc[df_anio['indice'].idxmin()]
    pais_max = df_anio.loc[df_anio['indice'].idxmax()]
    resultados_for.append({
        'anio': anio,
        'pais_mas_libertad': pais_min['pais'],
        'indice_min': pais_min['indice'],
        'pais_menos_libertad': pais_max['pais'],
        'indice_max': pais_max['indice']
    })

df_for = pd.DataFrame(resultados_for)
print(df_for)

# b) Enfoque vectorizado con groupby
print("\n" + "=" * 60)
print("b) Enfoque vectorizado con groupby")
print("=" * 60)
# idxmin/idxmax pueden devolver NaN si un grupo no tiene valores → los filtramos
idx_min_anio = df_america.groupby('anio')['indice'].idxmin().dropna().astype(int)
idx_max_anio = df_america.groupby('anio')['indice'].idxmax().dropna().astype(int)

df_min = df_america.loc[idx_min_anio, ['anio', 'pais', 'indice']].rename(
    columns={'pais': 'pais_mas_libertad', 'indice': 'indice_min'}
).reset_index(drop=True)
df_max = df_america.loc[idx_max_anio, ['anio', 'pais', 'indice']].rename(
    columns={'pais': 'pais_menos_libertad', 'indice': 'indice_max'}
).reset_index(drop=True)

df_groupby = pd.merge(df_min, df_max, on='anio')
print(df_groupby)

Total de registros válidos para América: 407
Países encontrados: 29

a) Enfoque con ciclo FOR
    anio  pais_mas_libertad  indice_min pais_menos_libertad  indice_max
0   2001             Canadá        0.80                Cuba       90.30
1   2002  Trinidad y Tobago        1.00                Cuba       97.83
2   2003  Trinidad y Tobago        2.00           Argentina    35826.00
3   2004  Trinidad y Tobago        2.00                Cuba       87.00
4   2005            Bolivia        4.50                Cuba       95.00
5   2006             Canadá        4.88                Cuba       96.17
6   2007             Canadá        3.33                Cuba       88.33
7   2008             Canadá        3.70                Cuba       94.00
8   2009     Estados Unidos        6.75                Cuba       78.00
9   2012            Jamaica        9.88                Cuba       71.64
10  2013            Jamaica       10.90                Cuba       70.92
11  2014             Canadá       10.99   

### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [5]:
# 4. Análisis anual del índice por país

# Tabla dinámica: filas = países, columnas = años, valores = indice máximo
pivot = pd.pivot_table(
    df,
    index='pais',
    columns='anio',
    values='indice',
    aggfunc='max',
    fill_value=0
)
print("=" * 60)
print("PIVOT TABLE: indice máximo por país y año")
print("=" * 60)
print(pivot.head(10))
print(f"\nDimensiones: {pivot.shape}")

# a) País con mayor y menor indice en la tabla
print("\n" + "=" * 60)
print("a) Valores extremos en toda la tabla")
print("=" * 60)
valor_max = pivot.values.max()
valor_min_no_cero = pivot.replace(0, np.nan).min().min()

pais_max_idx = pivot.stack().idxmax()
pais_min_idx = pivot.replace(0, np.nan).stack().idxmin()

print(f"Mayor indice: {valor_max:.4f} → País: {pais_max_idx[0]}, Año: {pais_max_idx[1]}")
print(f"Menor indice (no cero): {valor_min_no_cero:.4f} → País: {pais_min_idx[0]}, Año: {pais_min_idx[1]}")

# b) Años con indice promedio más alto/bajo
print("\n" + "=" * 60)
print("b) Años con promedios extremos")
print("=" * 60)
promedios_anios = pivot.replace(0, np.nan).mean(axis=0).sort_values(ascending=False)
print("Top 5 años con mayor indice promedio:")
print(promedios_anios.head(5))
print("\nTop 5 años con menor indice promedio:")
print(promedios_anios.tail(5))

# c) País con mayor variabilidad
print("\n" + "=" * 60)
print("c) País con mayor variabilidad")
print("=" * 60)
pivot_nan = pivot.replace(0, np.nan)
variabilidad = (pivot_nan.max(axis=1) - pivot_nan.min(axis=1)).sort_values(ascending=False)
print("Top 10 países con mayor variabilidad:")
print(variabilidad.head(10))

# d) Países con indice constante
print("\n" + "=" * 60)
print("d) Países con indice constante")
print("=" * 60)
paises_constantes = variabilidad[variabilidad == 0].index.tolist()
print(f"Cantidad: {len(paises_constantes)}")
if paises_constantes:
    print(paises_constantes[:20])

# e) Países sin datos (todos los valores en 0)
print("\n" + "=" * 60)
print("e) Países sin datos (todos en 0)")
print("=" * 60)
paises_sin_datos = pivot[(pivot == 0).all(axis=1)].index.tolist()
print(f"Cantidad: {len(paises_sin_datos)}")
if paises_sin_datos:
    print(paises_sin_datos[:20])
    print("\nExplicación: Estos países pueden no haber sido evaluados en ningún año del dataset,")
    print("o haber quedado sin coincidencias tras el merge debido a códigos ISO no estándar.")

PIVOT TABLE: indice máximo por país y año
anio               2001   2002      2003   2004   2005   2006   2007   2008  \
pais                                                                          
Afghanistán        35.5  40.17     28.25  39.17  44.25  56.50  59.25  54.25   
Albania             0.0   6.50     11.50  14.17  18.00  25.50  16.00  21.75   
Alemania            1.5   1.33      2.00   4.00   5.50   5.75   4.50   3.50   
Algeria            31.0  33.00     43.50  40.33  40.00  40.50  31.33  49.56   
Andorra             0.0   0.00      0.00   0.00   0.00   0.00   0.00   0.00   
Angola             30.2  28.00     26.50  18.00  21.50  26.50  29.50  36.50   
Antigua y Barbuda   0.0   0.00      0.00   0.00   0.00   0.00   0.00   0.00   
Arabia Saudita     62.5  71.50     79.17  66.00  76.00  59.75  61.75  76.50   
Argentina          12.0  15.17  35826.00  13.67  17.30  24.83  14.08  11.33   
Armenia             0.0  25.17     23.50  26.00  25.50  23.63  22.75  31.13   

anio     